# Cross-Axis Faceted HEDM (XAF-HEDM) — design & simulation demo

Far-field HEDM through a **cubic diamond anvil cell** with a ~15° opening on all six faces.
Narrow ω wedges are collected through the four equatorial faces per mounting; the cell is
**remounted about an orthogonal axis** so the top/bottom faces reach the equator, and both
mountings are **merged into one reciprocal-space reconstruction** (each fills the other's
missing cone). This notebook walks through the toolkit end to end.

It is a thin, differentiable layer over `midas-diffract`; every design number below comes
from the real forward model + autograd, not hand-waving.

In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')  # macOS OpenMP double-init guard
import numpy as np
from dataclasses import replace
from midas_xaf import XAFConfig, XAFForwardModel, make_sample, geometry, metrics
from midas_xaf import sweep as S, merge as MG, reconstruct as R
print('midas_xaf ready')

midas_xaf ready


## 1. Geometry, forward model, coverage

The exit cone caps 2θ at the opening half-angle, so a smaller opening forces a longer Lsd to
keep the accessible rings on the detector.

In [2]:
cfg = XAFConfig(material='zirconia_monoclinic', energy_keV=80.0,
                opening_full_deg=20.0, n_grains=40, seed=1)
print('geometry:', {k: (round(v, 3) if isinstance(v, float) else v)
                     for k, v in geometry.geometry_summary(cfg).items()})
fwd = XAFForwardModel(cfg)
grains = make_sample(cfg)
sim = fwd.simulate(grains)
spg = metrics.spots_per_grain(sim)
print(f'accessible spots = {len(sim.table)},  spots/grain median = {np.median(spg):.0f}')
print(f'Friedel completeness = {metrics.friedel_completeness(sim):.2f}  '
      '(opposite-face conjugacy -> grain COM from Friedel pairs is ~free)')

geometry: {'energy_keV': 80.0, 'wavelength_A': 0.155, 'opening_full_deg': 20.0, 'tth_max_deg': 10.0, 'wedge_half_deg': 10.0, 'Lsd_mm': 391.999, 'tth_at_detector_edge_deg': 11.085, 'detector_limited': False}
accessible spots = 17386,  spots/grain median = 436
Friedel completeness = 1.00  (opposite-face conjugacy -> grain COM from Friedel pairs is ~free)


## 2. Exit-path shadowing

As the sample rotates within a wedge, the diffracted beam must clear a face opening whose
transmitting cone **rotates with ω** — so different detector sectors go dark at different ω.
The exact `cone` gate vs the naive `2θ ≤ half` cap shows how much this matters.

In [3]:
sim_cap = XAFForwardModel(replace(cfg, exit_model='tth_cap')).simulate(grains)
print(f'cone (exact) : {len(sim.table)} spots')
print(f'tth_cap (naive): {len(sim_cap.table)} spots')
print(f'-> naive over-counts by {100*(len(sim_cap.table)/len(sim.table)-1):.0f}% '
      '(spots that never physically exit)')

cone (exact) : 17386 spots
tth_cap (naive): 27274 spots
-> naive over-counts by 57% (spots that never physically exit)


## 3. Strain sensitivity vs opening angle  (the 15° vs 20° v2-cell decision)

Worst-direction strain 1σ precision (a proper CRLB folding in detector/ω noise), single
mounting vs merged cross-axis. Lower is better. The plot is saved to disk (not shown inline).

In [4]:
base = XAFConfig(material='zirconia_monoclinic', energy_keV=80.0, n_grains=15, seed=3)
rows = S.sweep_opening(base, openings_deg=(10, 12, 15, 18, 20, 25, 30), verbose=False)
print(f"{'open':>4} {'spots/gr':>8} {'strain_single':>13} {'strain_merged':>13} {'xgain':>5}")
for r in rows:
    print(f"{r['_swept_value']:4.0f} {r['median_spots_per_grain']:8.0f} "
          f"{r['strain_precision_ue_single']:13.1f} {r['strain_precision_ue_merged']:13.1f} "
          f"{r['precision_gain']:5.2f}")
out = S.plot_opening_sweep(rows, 'strain_vs_opening_demo.png')
print('saved plot ->', os.path.abspath(out))

open spots/gr strain_single strain_merged xgain
  10       26        2764.9         895.7  3.09
  12       44        1121.7         552.9  2.03
  15      132         635.6         280.9  2.26
  18      286         400.7         189.1  2.12
  20      424         334.0         147.8  2.26
  25     1096         201.7          92.7  2.18
  30     2280         137.5          63.6  2.16


saved plot -> /Users/hsharma/opt/MIDAS/packages/midas_xaf/examples/strain_vs_opening_demo.png


## 4. Cross-axis merge gain

How much the orthogonal second mounting improves strain determinability over one mounting.

In [5]:
gain = metrics.cross_axis_gain(fwd, grains)
print('strain precision (µε): single {:.0f} -> merged {:.0f}  (x{:.2f})'.format(
    gain['strain_precision_ue_single'], gain['strain_precision_ue_merged'],
    gain['precision_gain']))
print('full-rank fraction: single {:.2f}, merged {:.2f}'.format(
    gain['frac_full_rank_single'], gain['frac_full_rank_merged']))

strain precision (µε): single 331 -> merged 150  (x2.20)
full-rank fraction: single 1.00, merged 1.00


## 5. Beam modes & grain-position localization

Box beam localizes grains only via the diffraction geometry (Friedel); a point/line pencil
beam localizes them directly to ~`beam_size/√12`.

In [6]:
for mode in ('box', 'line'):
    c = replace(cfg, beam_mode=mode, beam_size_um=(3.0 if mode != 'box' else 2.0), n_grains=12)
    f = XAFForwardModel(c)
    loc = metrics.position_localization(f, make_sample(c))
    print(f"{mode:5s}: box-Friedel={loc['box_friedel_position_um']:.2f} µm  "
          f"scanning={loc['scanning_position_um']:.2f} µm  "
          f"effective={loc['effective_position_um']:.2f} µm")

box  : box-Friedel=5.03 µm  scanning=0.58 µm  effective=5.03 µm


line : box-Friedel=5.03 µm  scanning=0.87 µm  effective=0.87 µm


## 6. Fiducial remount registration

Embedded high-Z markers recover the rigid remount transform. Three non-collinear markers are
the minimum (two are rotationally degenerate); accuracy is set by how well the markers are
localized — which favors point-focus scanning.

In [7]:
fr = MG.fiducial_registration_study(fwd, grains, n_fiducials_list=(2, 3, 4, 6), trials=15)
print(f"marker localization sigma = {fr['sigma_um']:.2f} µm")
for r in fr['rows']:
    tag = ' (degenerate)' if r['degenerate'] else ''
    print(f"  {r['n_fiducials']} markers: median remount angle error = "
          f"{r['median_angle_error_deg']:.2f}°{tag}")

marker localization sigma = 5.04 µm
  2 markers: median remount angle error = 56.98° (degenerate)
  3 markers: median remount angle error = 29.21°
  4 markers: median remount angle error = 15.64°
  6 markers: median remount angle error = 10.02°


## 7. Merged reconstruction round-trip

Correspondence-known Levenberg–Marquardt refine (Huber-robust to near-Ewald-tangency spots),
seeded near truth as an indexer would. Compares single-mounting vs merged recovery under noise.

In [8]:
st = R.recovery_study(fwd, grains, n_grains=10, noise=True, seed=0)
print('NOISY recovery, single -> merged:')
print(f"  orientation: {st['median_misori_deg_single']*1000:.1f} -> "
      f"{st['median_misori_deg_merged']*1000:.1f} millideg")
print(f"  position:    {st['median_pos_err_um_single']:.2f} -> "
      f"{st['median_pos_err_um_merged']:.2f} µm")
print(f"  strain:      {st['median_strain_err_ue_single']:.0f} -> "
      f"{st['median_strain_err_ue_merged']:.0f} µε")

NOISY recovery, single -> merged:
  orientation: 15.1 -> 11.6 millideg
  position:    7.88 -> 6.27 µm
  strain:      139 -> 84 µε


## 8. Reciprocal-space coverage

The signature figure: which scattering-vector **directions** are accessible in the common sample
frame, per mounting (blue = mounting 1, orange = mounting 2, green = both). The two orthogonal
**blind cones** and the cross-axis merge filling them are directly visible.

Be honest about the limit: even merged, only a fraction of reciprocal space is reached — the
narrow 4-wedge access + small exit cone leave a severe missing region. The merge is a real
~1.5× gain, not a cure. Larger openings recover more (the panel), which is the v2-cell argument.

In [9]:
from midas_xaf import coverage as C

dirs, lab = C.direction_coverage(cfg, n_dirs=6000, n_shells=5)
fr = C.coverage_fraction(lab)
print('direction coverage: single {single_mounting:.0%}, '
      'merged {merged:.0%}  (cross-axis gain {gain:.2f})'.format(**fr))

C.plot_coverage(dirs, lab, 'coverage_demo.png',
                title='reciprocal-space coverage (single vs merged)')
C.plot_opening_coverage_panel(cfg, [10, 15, 20, 30], 'coverage_vs_opening_demo.png')
print('saved: coverage_demo.png, coverage_vs_opening_demo.png')

direction coverage: single 24%, merged 36%  (cross-axis gain 1.49)


saved: coverage_demo.png, coverage_vs_opening_demo.png


## 9. One vs two vs three mountings

The first orthogonal mounting (1→2) fills the blind cone — the core of the technique. A third
mounting with the rotation axis along the remaining crystal direction (axes along all three) adds
~1.3× strain precision + registration redundancy, but little extra coverage (the missing region is
wedge-limited, not blind-cone-limited).

In [10]:
from dataclasses import replace
from midas_xaf import metrics

ORTHO = (((1., 0, 0), 90.0), ((0, 1., 0), 90.0))   # R_x(90) for M2, R_y(90) for M3
cfg3 = replace(cfg, n_mountings=3, remount_specs=ORTHO)
fwd3 = XAFForwardModel(cfg3)
g3 = make_sample(cfg3)

for k in (1, 2, 3):
    det = metrics.population_strain_determinability(fwd3, g3, mountings=list(range(k)))
    _, labs = C.direction_coverage(replace(cfg3, n_mountings=k), n_dirs=6000, n_shells=5)
    print(f"{k} mounting(s): strain sigma = {det['median_strain_precision_ue']:.0f} ue, "
          f"coverage = {(labs > 0).mean():.0%}")

C.plot_mounting_progression(cfg3, 'mountings_1v2v3_demo.png')
print('saved: mountings_1v2v3_demo.png')

1 mounting(s): strain sigma = 331 ue, coverage = 24%


2 mounting(s): strain sigma = 150 ue, coverage = 36%


3 mounting(s): strain sigma = 115 ue, coverage = 40%
saved: mountings_1v2v3_demo.png


## 10. End-to-end analysis pipeline on the digital twin

The design metrics assume the grains can be found from the data. Here we test that on a realistic
**measured** spot list (true |F|² detection floor, centroid/ω noise, Pilatus module gaps, spurious
peaks, correspondence stripped): confirm orientation is uniquely indexable, then run the full
chain (assign → robust Levenberg–Marquardt refine) and score it against ground truth. This is the
day-1 analysis pipeline, validated before any beam time.

In [11]:
from dataclasses import replace
from midas_xaf import synth, indexing, pipeline

# small sample for a quick end-to-end demo
cfg_demo = replace(cfg, n_grains=8)
grains = make_sample(cfg_demo)

# Digital twin: ground-truth grains -> realistic MEASURED spot list (true |F|^2
# detection floor, centroid/omega noise, module gaps, spurious peaks, unlabelled).
d = synth.make_measured_spots(cfg_demo, grains, seed=1)
print('measured spots:', d['summary'])

# Is orientation uniquely determined by the sparse data? (go/no-go)
u = indexing.orientation_uniqueness(XAFForwardModel(cfg_demo), grains, d['spots'],
                                    n_random=100, seed=0)
print(f"indexable: {u['frac_indexable']:.0%}  "
      f"(true match {u['median_true_matched']:.0f} vs best-random {u['median_best_random']:.0f})")

# Full chain: twin -> assign -> robust refine, scored vs ground truth.
res = pipeline.run_pipeline(cfg_demo, grains, seed=1)
print(f"recovered {res.frac_recovered:.0%} of grains | "
      f"misorientation {res.median_misorientation_mdeg:.1f} mdeg | "
      f"strain err {res.median_strain_err_ue:.0f} ue | "
      f"assignment purity {res.median_assignment_purity:.0%}")

measured spots: {'n_accessible': 3538, 'n_detected': 2124, 'n_spurious': 212, 'n_total': 2336, 'detect_frac': 0.6, 'spurious_frac': 0.1}


indexable: 100%  (true match 246 vs best-random 6)


recovered 100% of grains | misorientation 17.6 mdeg | strain err 206 ue | assignment purity 100%


## 11. Delivered HPCAT samples — size, clean-merge regime, and commissioning geometry

The HPCAT team delivered two samples for XAF-HEDM: **Al$_2$O$_3$ (corundum)** and **pyrope garnet**, ~**1 mm cubes** with **10–15 µm grains**.

**The key design constraint is the merge.** XAF merges the *same* grains across orthogonal mountings, so the sample should ideally fit *entirely in the beam*. A far-field beam illuminates a **column through the full sample depth**, not a compact interior volume — so a 100×100 µm box on the intact 1 mm cube lights up a 100×100×1000 µm column of **~5,700 grains**, which garnet cannot index (>50% overlap). Shrinking the beam to a ~30 µm pencil makes the column indexable (~500 grains) but then two *orthogonal* columns intersect in only ~15 grains — nothing merges. **Indexability wants a small beam, the merge wants a large one; on an oversized sample you can't have both.**

**The third dimension must come from the sample.** Two clean routes: (i) **sub-sample a ~100 µm chip** and illuminate it whole with a **~120–150 µm square box beam** (slightly larger than the chip, so it stays lit through all rotations) — ~500 grains, indexable *and* fully mergeable, single-shot; or (ii) **scan + register** a ~30 µm pencil across each mounting and merge the common volume. There is *no* clean single-shot keyhole of the intact cube.

**Commissioning geometry (this beamtime):** Varex 4343CT CdTe (2880² px, 150 µm, monolithic) at **63 keV**, **900 mm**. For the 23° cell (2θ_max ≈ 11.5°) the optimal distance is ~956 mm (90% detector fill); **900 mm (85% fill) is essentially optimal** — moving to 956 mm buys only ~5% in strain precision.

In [ ]:
import numpy as np
from midas_xaf import XAFConfig, XAFForwardModel, make_sample, metrics

def varex_cfg(material, ng, Lsd_mm=900.0, energy=63.0, radius=50.0):
    """Commissioning config: Varex 4343CT, 23 deg cell, 3 mountings."""
    return XAFConfig(material=material, opening_full_deg=23.0, n_mountings=3,
        remount_specs=(((1.,0,0),90.),((0,1.,0),90.)), energy_keV=energy,
        px_um=150.0, n_pixels_y=2880, n_pixels_z=2880, detector_type="none",
        Lsd_um=(Lsd_mm*1000.0 if Lsd_mm else None),
        sigma_det_px=1.0, sigma_omega_steps=1.0,
        n_grains=ng, sample_radius_um=radius, strain_rms=1e-3, n_fiducials=0, seed=1)

# --- optimal distance (from an UNSET config; independent of energy) ---
c_opt = varex_cfg("garnet_pyrope", 10, Lsd_mm=None)
opt = c_opt.resolved_Lsd_um()/1000
fill900 = 100*900*np.tan(np.radians(c_opt.tth_max_deg))/216
print(f"2theta_max = {c_opt.tth_max_deg:.1f} deg  ->  optimal Lsd = {opt:.0f} mm "
      f"(90% fill);  900 mm = {fill900:.0f}% fill (essentially optimal)")

# --- clean-merge sample-size limit (fully-illuminated cube) ---
print("\nOverlap of a FULLY-illuminated cube vs edge (23 deg, 3 mounts):")
for grain in (10.0, 15.0):
    gvol = (4/3)*np.pi*(grain/2)**3
    row = []
    for L in (60, 80, 100, 130):
        ng = int(round(L**3/gvol))                 # honest count (no cap)
        ovs = {}
        for m in ("alumina","garnet_pyrope"):
            cfg = varex_cfg(m, ng, radius=L/2)
            ovs[m] = 100*metrics.spot_overlap(
                XAFForwardModel(cfg).simulate(make_sample(cfg)))["overlap_fraction"]
        row.append(f"{L}um(Al {ovs['alumina']:.0f}%/Grt {ovs['garnet_pyrope']:.0f}%)")
    print(f"  {grain:.0f} um grains: " + "  ".join(row))
print("  -> ~20% clean-merge ceiling: garnet ~110 um (15 um grains) / ~70 um (10 um)")

# --- working-point feasibility at 63 keV / 900 mm ---
print("\nFeasibility at 63 keV / 900 mm (~100 um sub-volume):")
for material in ("alumina","garnet_pyrope"):
    c = varex_cfg(material, 50, Lsd_mm=900.0)
    fwd = XAFForwardModel(c); g = make_sample(c); sim = fwd.simulate(g)
    det = metrics.population_strain_determinability(fwd, g)
    print(f"  {material:>14}: {np.median(metrics.spots_per_grain(sim)):5.0f} spots/grain, "
          f"{100*metrics.spot_overlap(sim)['overlap_fraction']:.1f}% overlap, "
          f"Friedel {metrics.friedel_completeness(sim):.2f}, "
          f"strain CRLB {det['median_strain_precision_ue']:.0f} ue")

### 11b. Detector dynamic range (Varex integrating panel)

The Varex 4343CT is charge-*integrating*, not photon-counting: saturation ~64000, background ~2000 → usable dynamic range only ~**32×**. Reflection |F|²·LP (midas-hkls structure factors) spans orders of magnitude, so at a single exposure only spots within ~32× of the brightest survive; the rest fall into the noise floor. The lost spots are the weak **high-Q** ones — which carry the most strain information — so the gate hurts strain precision *more* than the count loss implies.

**Result:** garnet keeps ~16% of reflections (huge |F|² span) → strain CRLB 44→**124 µε** (2.8× worse); alumina keeps ~51% → 134→**207 µε** (1.6× worse). Both stay fully determinable, and garnet still wins. **Mitigation:** multi-exposure HDR (short frame for bright spots + long frame for weak high-Q ones) recovers the range.

In [ ]:
from midas_xaf import metrics
# per-reflection survival under the Varex dynamic range (packaged helper)
for material in ("alumina", "garnet_pyrope"):
    c = varex_cfg(material, 4); fwd = XAFForwardModel(c)
    d = metrics.dynamic_range_survival(fwd, saturation_counts=64000,
                                       background_counts=2000, exposure_fill=0.90)
    span = d["intensities"][d["intensities"]>0]
    print(f"{material:>14}: {d['n_survive']:4d}/{d['n_total']} reflections survive "
          f"({100*d['fraction']:.0f}%);  |F|^2 span {span.max()/span.min():.1e}")
print("\n(strain-CRLB impact — full gated calc in dev/dynamic_range_impact.py:")
print("   alumina 134->207 ue (1.6x),  garnet 44->124 ue (2.8x); both feasible)")

## Takeaways

* **20° opening beats 15°** ~1.9× on strain precision (and 3.2× more spots/grain), even after the
  shorter-Lsd penalty — supports the v2 cell.
* **Cross-axis merge** buys ~2.2× strain determinability and clear gains in orientation/position
  recovery; the orthogonal blind cones are complementary.
* **Friedel completeness ≈ 1.0** from opposite-face conjugacy; **exit-path shadowing** is a real
  ~40% effect that must be modeled.
* **Point-focus** localizes grains ~5.8× better than Friedel-only, and is what makes **fiducial
  registration** (3+ markers) accurate.

**Caveats to keep honest:** max 2θ ≈ opening half-angle makes the full 6-component strain tensor
the hardest quantity even merged; the remount must preserve a rigid sample (watch martensitic /
hysteretic materials like zirconia under pressure); and this reconstructor is a local refiner —
global indexing supplies the seed.